# 03 — Imbalance Experiment

## Purpose
Evaluate all RAC scoring variants under genuine database-level minority class
scarcity, simulated by building subsampled ChromaDB collections at controlled
majority:minority ratios.

## Approach
For each ratio in `[2, 3, 5, 10]`, a pair of subsampled ChromaDB collections
is built from the fully populated source database. **Majority class records
(Blue, Black) are kept intact. Minority class records (Green, TTR) are
subsampled down** to `floor(min_majority_count / ratio)` per class.

All scoring variants are then evaluated against these subsampled collections
using the standard test set — no retrieval-time filtering is applied.

This design reflects real-world deployment: minority items are scarce because
they were never collected in sufficient quantities, not because majority items
were artificially removed. It is a strictly harder stress test for the DNDS
density correction than retrieval-time filtering, because the scoring function
must identify minority queries against a naturally majority-dominated neighborhood.

## Inputs
- `dataset/CVPR_2024_dataset_Test/` — test images
- `dataset_text/test.csv` — test labels and filenames
- `chroma_db/` — fully populated source collections (treated as read-only)

## Outputs
- `results/phase2/imbalance_results.json`
- `results/phase2/imbalance_summary.csv`
- `figures/phase2/minority_f1_vs_imbalance.png`

In [ ]:
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
import time
import numpy as np

sys.path.insert(0, str(Path("../..").resolve()))

from tqdm.auto import tqdm

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_class_counts,
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)
from src.phase2.evaluation import (
    compute_metrics,
    load_image_as_numpy,
    save_imbalance_summary_csv,
    save_results,
)
from src.phase2.gpu_utils import (
    clear_gpu_memory,
    get_device,
    maybe_periodic_gpu_maintenance,
    print_device_info,
    print_gpu_memory,
)
from src.phase2.imbalance import (
    infer_class_groups,
    build_imbalanced_collections,
    get_imbalanced_collection_names,
    teardown_imbalanced_collections,
)
from src.phase2.scoring import (
    global_dnds,
    idw,
    kde_dnds,
    local_dnds,
    majority_vote,
    traditional,
)
from src.phase2.traditional import load_phase1_traditional_components
from src.phase2.visualization import plot_minority_f1_vs_imbalance

CONFIG = get_phase2_config()

# Per-notebook GPU controls
PREFER_GPU = True
USE_HALF_PRECISION = False
CLEANUP_INTERVAL = 0
MEMORY_LOG_INTERVAL = 0

# Parallel imbalance controls
PARALLEL_IMBALANCE = True
IMBALANCE_MAX_WORKERS = 10

# Progress logging controls
SHOW_NOTEBOOK_PROGRESS = False
IMBALANCE_LOG_INTERVAL = 250

DEVICE = get_device(prefer_gpu=PREFER_GPU)

REPO_ROOT = Path("../..").resolve()
TEST_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Test"
TEST_CSV = REPO_ROOT / "dataset_text" / "test.csv"
RESULTS_PATH = REPO_ROOT / "results" / "phase2" / "imbalance_results.json"
IMBALANCE_CSV_PATH = REPO_ROOT / "results" / "phase2" / "imbalance_summary.csv"
IMBALANCE_LOG_PATH = REPO_ROOT / "results" / "phase2" / "imbalance_experiment.log"
FIG_PATH = REPO_ROOT / "figures" / "phase2" / "minority_f1_vs_imbalance.png"
IMBALANCE_DB_PATH = REPO_ROOT / "chroma_db_imbalanced"

IMBALANCE_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
IMBALANCE_LOG_PATH.write_text("", encoding="utf-8")

print_device_info(DEVICE)
if MEMORY_LOG_INTERVAL > 0:
    print_gpu_memory(prefix="Startup GPU memory", device=DEVICE)

print(f"Imbalance progress log: {IMBALANCE_LOG_PATH}")

In [ ]:
test_samples, missing_examples, total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

if missing_examples:
    print("Skipped test rows with missing image files (up to 10 shown):")
    for item in missing_examples:
        print(f"  - {item}")

if not test_samples:
    raise RuntimeError("No test samples found after image path resolution.")

print(
    f"Test samples available for imbalance experiment: {len(test_samples)} / {total_rows}"
)

client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)

image_class_counts = get_class_counts(image_collection)
majority_classes, minority_classes = infer_class_groups(
    image_class_counts, threshold=float(CONFIG["majority_threshold"])
)
CONFIG["majority_classes"] = majority_classes
CONFIG["minority_classes"] = minority_classes
print(
    f"Dynamic class grouping -> majority: {majority_classes}, minority: {minority_classes}"
)

image_ckpt = REPO_ROOT / "models" / "mobilenetv3_best.pth"
text_ckpt = REPO_ROOT / "text_models" / "distilbert_best.pth"
traditional_ready = False
traditional_kwargs = {}
if image_ckpt.exists() and text_ckpt.exists():
    clear_gpu_memory()
    traditional_kwargs = load_phase1_traditional_components(
        image_checkpoint_path=image_ckpt,
        text_checkpoint_path=text_ckpt,
        num_classes=len(CONFIG["class_names"]),
        device=DEVICE,
        use_half_precision=USE_HALF_PRECISION,
    )
    traditional_ready = True

variants = {
    "majority_vote": majority_vote,
    "idw": idw,
    "global_dnds": global_dnds,
    "local_dnds": local_dnds,
    "kde_dnds": kde_dnds,
}

# Separate ChromaDB client for imbalanced collections
# Source collections in chroma_db/ are never modified
imbalanced_client = get_persistent_client(str(IMBALANCE_DB_PATH))

print("\n=== Imbalance Experiment Setup ===")
print(f"Source DB class counts: {image_class_counts}")
print(f"Majority classes (kept intact):      {CONFIG['majority_classes']}")
print(f"Minority classes (to be subsampled): {CONFIG['minority_classes']}")
print(f"Imbalance ratios: {CONFIG['imbalance_ratios']}")

majority_min = min(
    image_class_counts[c] for c in CONFIG["majority_classes"] if c in image_class_counts
)
print("\nExpected minority targets per class:")
for ratio in CONFIG["imbalance_ratios"]:
    target = max(1, majority_min // ratio)
    print(f"  ratio={ratio}:1 -> {target} records per minority class")

In [ ]:
all_results = {
    "ratios": CONFIG["imbalance_ratios"],
    "variants": {
        k: {}
        for k in list(variants.keys()) + (["traditional"] if traditional_ready else [])
    },
}

# Set True to delete each imbalanced collection pair after evaluation
# to free disk space. Set False to keep them for debugging.
TEARDOWN_AFTER_RATIO = True

# Ratios are independent, so we can parallelize across ratios.
# On shared GPU systems, 2 workers is a practical starting point.
PARALLEL_BY_RATIO = True
RATIO_MAX_WORKERS = 2


def _log_progress(message: str) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with IMBALANCE_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(f"[{timestamp}] {message}\n")


def _evaluate_single_ratio(ratio: int) -> tuple[str, dict[str, dict]]:
    print(f"\n{'='*60}")
    print(f"RATIO {ratio}:1")
    print(f"{'='*60}")
    _log_progress(f"START ratio {ratio}:1")

    # Use a per-ratio client handle to avoid cross-thread collection handle reuse.
    ratio_client = get_persistent_client(str(IMBALANCE_DB_PATH))

    imb_image_col, imb_text_col = build_imbalanced_collections(
        client=ratio_client,
        source_image_collection=image_collection,
        source_text_collection=text_collection,
        majority_classes=CONFIG["majority_classes"],
        minority_classes=CONFIG["minority_classes"],
        ratio=ratio,
        image_collection_name=get_imbalanced_collection_names(ratio)[0],
        text_collection_name=get_imbalanced_collection_names(ratio)[1],
        seed=42,
    )

    ratio_results: dict[str, dict] = {}

    # Evaluate RAC variants against this ratio's imbalanced collections.
    for name, fn in variants.items():
        _log_progress(f"START {name} @ {ratio}:1")

        y_true: list[str] = []
        y_pred: list[str] = []
        latencies: list[float] = []

        iterator = (
            enumerate(tqdm(test_samples, desc=f"{name} @ {ratio}:1"), start=1)
            if not PARALLEL_BY_RATIO
            else enumerate(test_samples, start=1)
        )

        for sample_index, sample in iterator:
            query_image = load_image_as_numpy(sample["image_path"])
            t0 = time.perf_counter()
            pred = fn(
                query_image=query_image,
                query_text=sample["text"],
                image_collection=imb_image_col,
                text_collection=imb_text_col,
                config=CONFIG,
                alpha=CONFIG["alpha"],
            )
            latencies.append((time.perf_counter() - t0) * 1000.0)
            y_true.append(sample["label"])
            y_pred.append(pred)

            maybe_periodic_gpu_maintenance(
                step_index=sample_index,
                cleanup_interval=CLEANUP_INTERVAL,
                memory_log_interval=MEMORY_LOG_INTERVAL,
                device=DEVICE,
                log_prefix=f"{name} @ {ratio}:1",
            )

            if sample_index % IMBALANCE_LOG_INTERVAL == 0 or sample_index == len(
                test_samples
            ):
                _log_progress(
                    f"{name} @ {ratio}:1 progress {sample_index}/{len(test_samples)}"
                )

        metrics = compute_metrics(y_true, y_pred, CONFIG["class_names"])
        metrics["inference_time_ms"] = float(np.mean(latencies))
        ratio_results[name] = metrics
        _log_progress(f"DONE {name} @ {ratio}:1")

    if TEARDOWN_AFTER_RATIO:
        print(f"\nTearing down imbalanced collections for ratio {ratio}:1 ...")
        teardown_imbalanced_collections(ratio_client, ratio)

    _log_progress(f"DONE ratio {ratio}:1")
    return str(ratio), ratio_results


ratios = list(CONFIG["imbalance_ratios"])
if PARALLEL_BY_RATIO and len(ratios) > 1:
    workers = max(1, min(RATIO_MAX_WORKERS, len(ratios)))
    print(f"Running ratios in parallel with {workers} workers.")
    _log_progress(f"RUN ratios in parallel workers={workers}")

    with ThreadPoolExecutor(max_workers=workers) as pool:
        future_map = {
            pool.submit(_evaluate_single_ratio, ratio): ratio for ratio in ratios
        }
        for future in as_completed(future_map):
            ratio = future_map[future]
            ratio_key, ratio_metrics = future.result()
            for variant_name, metrics in ratio_metrics.items():
                all_results["variants"][variant_name][ratio_key] = metrics
            print(f"Completed ratio {ratio}:1")
            _log_progress(f"COMPLETED ratio {ratio}:1")
else:
    _log_progress("RUN ratios sequentially")
    for ratio in ratios:
        ratio_key, ratio_metrics = _evaluate_single_ratio(ratio)
        for variant_name, metrics in ratio_metrics.items():
            all_results["variants"][variant_name][ratio_key] = metrics

# Traditional variant uses Phase 1 model weights and does not depend on
# ChromaDB imbalance. Run once and replicate across ratios to avoid
# concurrent model inference contention and redundant compute.
if traditional_ready:
    _log_progress("START traditional (single run)")

    t_true: list[str] = []
    t_pred: list[str] = []
    t_latencies: list[float] = []

    for sample_index, sample in enumerate(
        tqdm(test_samples, desc="traditional (single run)"), start=1
    ):
        query_image = load_image_as_numpy(sample["image_path"])
        t0 = time.perf_counter()
        pred = traditional(
            query_image=query_image,
            query_text=sample["text"],
            image_collection=image_collection,
            text_collection=text_collection,
            config=CONFIG,
            alpha=CONFIG["alpha"],
            **traditional_kwargs,
        )
        t_latencies.append((time.perf_counter() - t0) * 1000.0)
        t_true.append(sample["label"])
        t_pred.append(pred)

        maybe_periodic_gpu_maintenance(
            step_index=sample_index,
            cleanup_interval=CLEANUP_INTERVAL,
            memory_log_interval=MEMORY_LOG_INTERVAL,
            device=DEVICE,
            log_prefix="traditional (single run)",
        )

        if sample_index % IMBALANCE_LOG_INTERVAL == 0 or sample_index == len(
            test_samples
        ):
            _log_progress(
                f"traditional (single run) progress {sample_index}/{len(test_samples)}"
            )

    trad_metrics = compute_metrics(t_true, t_pred, CONFIG["class_names"])
    trad_metrics["inference_time_ms"] = float(np.mean(t_latencies))
    for ratio in ratios:
        all_results["variants"]["traditional"][str(ratio)] = dict(trad_metrics)

    _log_progress("DONE traditional (single run)")

In [ ]:
import pandas as pd

save_results(all_results, str(RESULTS_PATH))
save_imbalance_summary_csv(all_results, str(IMBALANCE_CSV_PATH))
plot_minority_f1_vs_imbalance(all_results, str(FIG_PATH))

# if traditional_ready:
#     del traditional_kwargs
# clear_gpu_memory()

print(f"Saved JSON results: {RESULTS_PATH}")
print(f"Saved CSV summary: {IMBALANCE_CSV_PATH}")
print(f"Saved figure: {FIG_PATH}")

# Tabular summary for quick inspection
rows = []
for ratio in all_results.get("ratios", []):
    ratio_key = str(ratio)
    for variant_name, by_ratio in all_results.get("variants", {}).items():
        metrics = by_ratio.get(ratio_key, {})
        per_f1 = metrics.get("per_class_f1", {})
        rows.append(
            {
                "ratio": ratio,
                "variant": variant_name,
                "accuracy": round(float(metrics.get("accuracy", 0.0)), 4),
                "macro_f1": round(float(metrics.get("macro_f1", 0.0)), 4),
                "weighted_f1": round(float(metrics.get("weighted_f1", 0.0)), 4),
                "green_f1": round(float(per_f1.get("Green", 0.0)), 4),
                "ttr_f1": round(float(per_f1.get("TTR", 0.0)), 4),
            }
        )

summary_df = pd.DataFrame(rows).sort_values(["ratio", "variant"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("IMBALANCE RESULTS SUMMARY TABLE")
print("=" * 80)
print(summary_df)